# 05 Critic Agent And Self Correction

In [1]:
# ==================================================
# Notebook 05
# Critic Agent & Self-Correction
# ==================================================

from typing import TypedDict
from typing import List
from typing import Dict

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from langgraph.graph import StateGraph, END

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
class ResearchState(TypedDict):

    research_goal: str

    queries_executed: List[str]

    raw_findings: List[str]

    deduplicated_findings: List[str]

    conflicts: List[str]

    critic_score: float

    draft_report: str

    final_report: str

    iteration_count: int

    status: str

    errors: List[str]

    metadata: Dict

In [4]:
sample_state = {
    "research_goal": "Compare cloud costs against competitors",
    "queries_executed": [],
    "raw_findings": [],
    "deduplicated_findings": [
        "AWS pricing increased 8%",
        "Azure pricing remained stable",
    ],
    "conflicts": [],
    "critic_score": 0.0,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "errors": [],
    "metadata": {},
}

In [5]:
goal_embedding = embedding_model.encode(sample_state["research_goal"])

In [6]:
findings_text = " ".join(sample_state["deduplicated_findings"])

In [7]:
findings_embedding = embedding_model.encode(findings_text)

In [8]:
coverage_score = cosine_similarity(
    goal_embedding.reshape(1, -1), findings_embedding.reshape(1, -1)
)[0][0]

coverage_score

0.4053526

In [9]:
def critic_agent(state: ResearchState):

    print("\nCritic Agent Running...")

    goal = state["research_goal"]

    findings = " ".join(state["deduplicated_findings"])

    goal_emb = embedding_model.encode(goal)

    findings_emb = embedding_model.encode(findings)

    score = cosine_similarity(goal_emb.reshape(1, -1), findings_emb.reshape(1, -1))[0][
        0
    ]

    state["critic_score"] = float(score)

    state["status"] = "critic_completed"

    return state

In [10]:
updated_state = critic_agent(sample_state)

updated_state["critic_score"]


Critic Agent Running...


0.4053525924682617

In [11]:
def explain_score(score):

    if score >= 0.80:
        return "Excellent Coverage"

    elif score >= 0.65:
        return "Moderate Coverage"

    else:
        return "Insufficient Coverage"

In [12]:
explain_score(updated_state["critic_score"])

'Insufficient Coverage'

In [13]:
def critic_decision(state):

    score = state["critic_score"]

    iterations = state["iteration_count"]

    if iterations >= 3:

        return "writer"

    if score >= 0.75:

        return "writer"

    return "researcher"

In [14]:
def researcher_agent(state):

    print("\nResearch Agent Running...")

    state["iteration_count"] += 1

    if state["iteration_count"] == 1:

        state["deduplicated_findings"].extend(
            ["AWS pricing increased 8%", "Azure pricing remained stable"]
        )

    elif state["iteration_count"] == 2:

        state["deduplicated_findings"].extend(["Google Cloud storage reduced 4%"])

    elif state["iteration_count"] == 3:

        state["deduplicated_findings"].extend(["Oracle Cloud increased 3%"])

    return state

In [15]:
def writer_agent(state):

    print("\nWriter Agent Running...")

    report = "\n".join(state["deduplicated_findings"])

    state["final_report"] = report

    state["status"] = "completed"

    return state

In [16]:
graph = StateGraph(ResearchState)

In [17]:
graph.add_node("researcher", researcher_agent)

graph.add_node("critic", critic_agent)

graph.add_node("writer", writer_agent)

In [18]:
graph.set_entry_point("researcher")

In [19]:
graph.add_edge("researcher", "critic")

In [20]:
graph.add_conditional_edges("critic", critic_decision)

In [21]:
graph.add_edge("writer", END)

In [22]:
app = graph.compile()

In [23]:
initial_state = {
    "research_goal": "Compare cloud costs against competitors",
    "queries_executed": [],
    "raw_findings": [],
    "deduplicated_findings": [],
    "conflicts": [],
    "critic_score": 0.0,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "errors": [],
    "metadata": {},
}

In [24]:
result = app.invoke(initial_state)


Research Agent Running...

Critic Agent Running...

Research Agent Running...

Critic Agent Running...

Research Agent Running...

Critic Agent Running...

Writer Agent Running...


In [25]:
print(result["final_report"])

AWS pricing increased 8%
Azure pricing remained stable
Google Cloud storage reduced 4%
Oracle Cloud increased 3%
